# Notebook de obtención, limpieza y transformación de datos — Fase 2

**Proyecto transversal · MCDI500 Programación para la Ciencia de Datos**
**Magíster en Ciencia de Datos e Inteligencia Artificial · Universidad Andrés Bello**

**Grupo 5 — Factores asociados a la duración autorizada de los permisos de uso de vía
pública en San Francisco**

## Introducción

Este cuaderno toma el archivo original de permisos y deja un conjunto validado, listo para
modelar la duración autorizada en la Fase 3.

**Lo que recibe de la Fase 1.** `docs/diccionario_variables.csv` y `docs/metadatos_fase1.json`:
qué variable es el objetivo, cuáles son explicativas, cuáles están prohibidas por contener la
respuesta, y cuál agrupa la partición. Esta fase **no vuelve a decidirlo**; lo lee y se detiene
si no lo encuentra.

**Lo que entrega.** `data/processed/permisos_limpio.csv` —interpretable, para el análisis y los
gráficos de la Fase 4—, `data/processed/permisos_procesado.csv` —codificado y escalado, para el
modelado de la Fase 3—, los transformadores ajustados en `src/`, y los anexos del informe en
`docs/anexos/`.

> **Un principio gobierna el orden de los apartados.** La partición va **antes** de cualquier
> ajuste de imputador, codificador o escalador. Calcular una mediana, una moda o una escala
> sobre el conjunto completo filtra al conjunto de prueba información que no debería conocer.
> El orden no es estético: es la diferencia entre una evaluación honesta y una inflada.

### Librerías utilizadas

Cada importación responde a una necesidad concreta. `GroupShuffleSplit` particiona respetando
los grupos; `OneHotEncoder` codifica categorías sin inventar un orden; los tres escaladores se
comparan entre sí en el apartado 5 antes de elegir uno.

In [1]:
import io, contextlib                # capturar salidas para los anexos del informe
import json                          # contrato con la Fase 1 y metadatos
import pickle                        # persistir los transformadores ajustados
import sys
import warnings                      # tratar los avisos de parseo como evidencia
from datetime import date
from pathlib import Path

import numpy as np                   # cálculo numérico y enmascaramiento aleatorio
import pandas as pd                  # estructuras tabulares
import matplotlib.pyplot as plt      # gráficos de exploración y control

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import (OneHotEncoder, StandardScaler, MinMaxScaler,
                                   RobustScaler, LabelEncoder, OrdinalEncoder)

# Reproducibilidad: la misma semilla que el cuaderno de la Fase 1.
SEMILLA = 42
np.random.seed(SEMILLA)


def capturar(funcion, *args, **kwargs):
    """
    Ejecuta una funcion, muestra su salida en pantalla y ademas la devuelve como texto.

    Sirve para que los anexos del informe se generen desde la misma ejecucion que produce
    los resultados, en lugar de copiarse a mano desde una captura de pantalla.
    """
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        resultado = funcion(*args, **kwargs)
    texto = buffer.getvalue()
    print(texto, end="")
    return resultado, texto


print("Cuaderno de la Fase 2 ·", date.today().isoformat())

Cuaderno de la Fase 2 · 2026-09-16


## Configuración del entorno y de las rutas

Las rutas se resuelven igual que en la Fase 1: subiendo hasta encontrar un marcador de
repositorio, en lugar de suponer que la raíz coincide con el directorio desde el que se lanza
el cuaderno. Así ambos cuadernos escriben en el mismo árbol de archivos aunque vivan en
carpetas distintas.

In [2]:
MARCADORES = (".git", "requirements.txt", ".gitignore")


def localizar_raiz(inicio=None, marcadores=MARCADORES, niveles=5, respaldo=True):
    """Resuelve la raiz del proyecto. Identica a la de la Fase 1."""
    actual = Path(inicio or Path.cwd()).resolve()
    for candidata in [actual, *actual.parents][:niveles + 1]:
        if any((candidata / m).exists() for m in marcadores):
            return candidata, "estructurado"
    if respaldo:
        return actual, "plano"
    raise FileNotFoundError(f"No se encontro marcador de repositorio desde {actual}.")


RAIZ, MODO = localizar_raiz()
DIR_CRUDO = RAIZ / "data" / "raw"
DIR_PROCESADO = RAIZ / "data" / "processed"
DIR_DOCS = RAIZ / "docs"
DIR_SRC = RAIZ / "src"
DIR_ANEXOS = DIR_DOCS / "anexos"
for d in (DIR_PROCESADO, DIR_DOCS, DIR_SRC, DIR_ANEXOS):
    d.mkdir(parents=True, exist_ok=True)

# Modulo compartido, generado por el cuaderno de la Fase 1.
if str(DIR_SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(DIR_SRC.resolve()))
try:
    import utilidades
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "No se encontro src/utilidades.py. Ejecute primero el cuaderno de la Fase 1."
    ) from e

print("Raíz del proyecto :", RAIZ)
print("Modo detectado    :", MODO)
print("Módulo importado  :", Path(utilidades.__file__).relative_to(RAIZ).as_posix())

Raíz del proyecto : /Users/ricardo/Desktop/Repositorios/proyecto-grupo5-mcdi500
Modo detectado    : estructurado
Módulo importado  : src/utilidades.py


### Procedencia del conjunto de datos y contrato con la Fase 1

La Fase 1 dejó escritas las decisiones sobre qué variable es el objetivo y cuáles no pueden
usarse para explicarla. Este cuaderno las lee. Si los artefactos no están, se detiene: es una
dependencia dura, no una sugerencia, porque sin el diccionario no se sabe qué variable contiene
la respuesta.

In [3]:
RUTA_DICC = utilidades.buscar_archivo("diccionario_variables.csv", RAIZ)
RUTA_META = utilidades.buscar_archivo("metadatos_fase1.json", RAIZ)

if RUTA_DICC is None or RUTA_META is None:
    raise FileNotFoundError(
        "Faltan los artefactos de la Fase 1. Ejecute S1_F1_Definicion.ipynb primero."
    )

DECLARADO = pd.read_csv(RUTA_DICC)                       # diccionario de la Fase 1
META_F1 = json.loads(Path(RUTA_META).read_text(encoding="utf-8"))

OBJETIVO     = META_F1["objetivo"]
EXPLICATIVAS = META_F1["explicativas"]
GRUPO_VAR    = META_F1["variable_grupo"]
PROHIBIDAS   = META_F1["prohibidas"]
INFORMATIVAS = META_F1["ausencia_informativa"]
FUENTE       = META_F1["fuente"]

print("Fuente   :", FUENTE["titulo"], f"({FUENTE['identificador']})")
print("Licencia :", FUENTE["licencia"])
print("Corte    :", FUENTE["corte_declarado"])
print("\nContrato leído de la Fase 1")
print(f"  objetivo             : {OBJETIVO}")
print(f"  explicativas ({len(EXPLICATIVAS)})     : {', '.join(EXPLICATIVAS)}")
print(f"  variable de grupo    : {GRUPO_VAR}")
print(f"  prohibidas           : {', '.join(PROHIBIDAS)}")
print(f"  ausencia informativa : {', '.join(INFORMATIVAS)}")

Fuente   : Active Street-Use Permits (x8nh-xzn6)
Licencia : Open Data Commons Public Domain Dedication and License (PDDL) 1.0
Corte    : 2026-09-09

Contrato leído de la Fase 1
  objetivo             : duracion_dias
  explicativas (11)     : tipo_permiso, analysis_neighborhood, agente, anio_aprobacion, distrito, n_segmentos, lag_aprob_inicio, mes_inicio, Permit Purpose, Inspector, permit_zipcode
  variable de grupo    : permit_number
  prohibidas           : Status, permit_start_date, permit_end_date
  ausencia informativa : Permit Purpose, Inspector


## 1. Obtención de los datos

La carga se encapsula en una función en lugar de una llamada suelta, por dos motivos: puede
probarse, y deja en un solo lugar la decisión de **leer todo como texto**.

Esa decisión es específica de esta fuente. Las coordenadas usan coma decimal y una parte de las
fechas admite dos lecturas, de modo que la inferencia automática de tipos puede acertar o
fallar según qué valor encuentre primero. Leer como texto obliga a declarar cada conversión, que
es lo que hace el apartado 3.

In [4]:
def cargar_datos(ruta, obligatorias=None):
    """
    Carga el archivo de permisos sin inferencia de tipos.

    Parametros
    ----------
    ruta : str | Path
        Archivo CSV a cargar.
    obligatorias : iterable of str | None
        Columnas que deben existir; si falta alguna se informa cual.

    Retorna
    -------
    pd.DataFrame
        Todas las columnas como texto.

    Lanza
    -----
    FileNotFoundError
        Si el archivo no existe.
    KeyError
        Si falta alguna columna obligatoria.
    """
    ruta = Path(ruta)
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontro el archivo {ruta}.")
    datos = pd.read_csv(ruta, dtype=str)
    if obligatorias:
        faltan = set(obligatorias) - set(datos.columns)
        if faltan:
            raise KeyError(f"Faltan columnas obligatorias: {sorted(faltan)}")
    return datos


ARCHIVO = utilidades.buscar_archivo(FUENTE["archivo"], RAIZ)
if ARCHIVO is None:
    raise FileNotFoundError(
        f"No se encontro {FUENTE['archivo']} en {DIR_CRUDO.relative_to(RAIZ).as_posix()}."
    )

df = cargar_datos(ARCHIVO, obligatorias=["permit_number", "permit_start_date",
                                         "permit_end_date", "Permit Type"])
df_crudo = df.copy(deep=True)          # referencia intacta para la validacion final

print("Archivo    :", ARCHIVO.relative_to(RAIZ).as_posix())
print("Dimensiones:", df.shape)

Archivo    : data/raw/Active_Street-Use_Permits_20260909.csv
Dimensiones: (6852, 29)


Mostramos las primeras filas para confirmar visualmente que las columnas se cargaron con el
nombre esperado. `head()` es la comprobación más barata que existe: si el separador o la
codificación fueran incorrectos, se vería aquí y no veinte celdas más abajo.

In [5]:
df.head()

,permit_number,streetname,Cross Street 1,Cross Street 2,Permit Type,Agent,AgentPhone,Permit Purpose,Approved Date,Status,...,Y,Latitude,Longitude,bpa,Location,the_geom,analysis_neighborhood,supervisor_district,data_as_of,data_loaded_at
0,26EXC-01641,08TH ST,NATOMA ST,HOWARD ST,Excavation,Ronan Construction Inc.,415-779-5262,"8th St, Clay St and Leavenworth St - Pavement ...",04/15/2026,APPROVED,...,"2110825,76329","37,77644115815","-122,4118820325",NaN,POINT (-122.41188203249555 37.77644115815177),POINT (-122.411882032 37.776441158),South of Market,6,2026/04/16 03:34:45 AM,2026/04/17 05:23:32 AM
1,25EXC-01889,SAN BRUNO AVE,MANSELL ST \ SALINAS AVE,ORDWAY ST,Excavation,"Cratus, Inc.",(415)939-2840,WD-2876 8-Inch Ductile Iron Water Main Repl...,07/18/2025,ACTIVE,...,"2090569,998","37,7209782871","-122,40087332838",NaN,POINT (-122.40087332838284 37.720978287099676),POINT (-122.400873328 37.720978287),Portola,10,2025/09/10 05:11:34 AM,2025/09/12 05:08:57 AM
2,23IE-00344,WALNUT ST,PACIFIC AVE,JACKSON ST,StrtImprov,"Partner's Contracting, Inc.",415-492-9233,Remove and reconstruct sidewalk and existing (...,08/26/2025,APPROVED,...,"2116443,35732","37,79125689282","-122,44945871865",202210184608,POINT (-122.44945871865453 37.79125689282032),POINT (-122.449458719 37.791256893),Presidio Heights,2,2025/08/28 03:42:52 AM,2025/08/29 05:13:53 AM
3,25EXC-01889,CAMPBELL AVE,SAN BRUNO AVE,BRUSSELS ST,Excavation,"Cratus, Inc.",(415)939-2840,WD-2876 8-Inch Ductile Iron Water Main Repl...,07/18/2025,ACTIVE,...,"2088480,85453","37,71525239967","-122,40007624052",NaN,POINT (-122.40007624051567 37.715252399671314),POINT (-122.400076241 37.7152524),Visitacion Valley,10,2025/09/10 05:11:34 AM,2025/09/12 05:08:57 AM
4,25EXC-01889,CANDLESTICK COVE WAY,EXECUTIVE PARK BLVD,NaN,Excavation,"Cratus, Inc.",(415)939-2840,WD-2876 8-Inch Ductile Iron Water Main Repl...,07/18/2025,ACTIVE,...,"2087023,572266","37,71134702909","-122,39401110729",NaN,POINT (-122.39401110729395 37.71134702909027),POINT (-122.394011107 37.711347029),Bayview Hunters Point,10,2025/09/10 05:11:34 AM,2025/09/12 05:08:57 AM
